# Autoresearch IRT Experiment Analysis

Analysis of autonomous prompt engineering results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, retained, retention_rate, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["retained"] = pd.to_numeric(df["retained"], errors="coerce")
df["retention_rate"] = pd.to_numeric(df["retention_rate"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    rate = row["retention_rate"]
    desc = row["description"]
    print(f"  #{i:3d}  retention={rate:.3f}  retained={int(row['retained'])}  {desc}")

## Retention Rate Over Time

Track how the best (kept) retention_rate evolves as experiments progress. The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_rate = valid.loc[0, "retention_rate"]

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["retention_rate"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["retention_rate"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line (higher is better)
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_rate = valid.loc[kept_mask, "retention_rate"]
running_max = kept_rate.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, rate in zip(kept_idx, kept_rate):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, rate),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Retention Rate (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch IRT Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_rate = df.iloc[0]["retention_rate"]
best_rate = kept["retention_rate"].max()
best_row = kept.loc[kept["retention_rate"].idxmax()]

print(f"Baseline retention_rate: {baseline_rate:.3f}")
print(f"Best retention_rate:     {best_rate:.3f}")
print(f"Total improvement:       {best_rate - baseline_rate:+.3f}")
print(f"Best experiment:         {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: retention={row['retention_rate']:.3f}  retained={int(row['retained'])}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's retention_rate
kept = df[df["status"] == "KEEP"].copy()
kept["prev_rate"] = kept["retention_rate"].shift(1)
kept["delta"] = kept["retention_rate"] - kept["prev_rate"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Rate':>8}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.3f}  {row['retention_rate']:.3f}     {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.3f}  {'':>8}  TOTAL improvement over baseline")